<a href="https://colab.research.google.com/github/ACJ08/flyrank-ml-internship/blob/main/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ACJ08/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
# Your experiment here
# ============================================================
# WEEK 2 - YOUR EXPERIMENT
# ============================================================
# Goal:
# 1. Test different decision-tree depths: 2, 3, and 4
# 2. Compare different feature sets
# 3. Compare in-sample results with a train/test split
# 4. Check which feature the tree chooses for its first split
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT THE TOOLS WE NEED
# ------------------------------------------------------------

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 2. DEFINE OUR ORIGINAL FEATURES
# ------------------------------------------------------------
# These are the features we are allowed to use.
# They represent information that can be observed before
# making a prediction about whether a page is declining.

original_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]


# ------------------------------------------------------------
# 3. CREATE X AND y
# ------------------------------------------------------------
# X = input features given to the model
# y = target/answer we want the model to predict
#
# y should already exist from the earlier notebook cell:
#
# is_declining_label = 1 -> page is declining
# is_declining_label = 0 -> page is not declining

X_original = (
    df[original_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"].values


print("Dataset shape:", X_original.shape)
print("Number of pages:", len(y))
print()


# ============================================================
# EXPERIMENT A
# Test different tree depths
# ============================================================

print("=" * 70)
print("EXPERIMENT A: TEST DIFFERENT TREE DEPTHS")
print("=" * 70)

# We will test three different tree depths.
# max_depth controls how complicated the decision tree can become.

depths = [2, 3, 4]

depth_results = []


for depth in depths:

    # --------------------------------------------------------
    # Create the decision tree
    # --------------------------------------------------------
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    # --------------------------------------------------------
    # Train the model using the full dataset
    # --------------------------------------------------------
    # NOTE:
    # This is IN-SAMPLE evaluation because we train and
    # evaluate using the same data.
    tree.fit(X_original, y)

    # --------------------------------------------------------
    # Get probability that each page is declining
    # --------------------------------------------------------
    # [:, 1] means:
    # "Give me the probability of class 1"
    #
    # Class 1 = declining
    tree_score = tree.predict_proba(X_original)[:, 1]

    # --------------------------------------------------------
    # Calculate Precision@20 and Precision@50
    # --------------------------------------------------------
    precision20 = precision_at_k(
        tree_score,
        y,
        20
    )

    precision50 = precision_at_k(
        tree_score,
        y,
        50
    )

    # Save the results
    depth_results.append({
        "max_depth": depth,
        "Precision@20": precision20,
        "Precision@50": precision50
    })

    # --------------------------------------------------------
    # Print the results
    # --------------------------------------------------------
    print()
    print(f"--- Decision Tree: max_depth={depth} ---")

    print(f"Precision@20: {precision20:.3f}")
    print(f"Precision@50: {precision50:.3f}")

    # --------------------------------------------------------
    # Print the actual tree
    # --------------------------------------------------------
    # This lets us READ the model as if/else rules.
    print()
    print("Readable tree:")
    print(
        export_text(
            tree,
            feature_names=original_features
        )
    )


# ------------------------------------------------------------
# Show all depth results in one table
# ------------------------------------------------------------

depth_results_df = pd.DataFrame(depth_results)

print()
print("=" * 70)
print("DEPTH COMPARISON")
print("=" * 70)

print(depth_results_df.to_string(index=False))


# ============================================================
# EXPERIMENT B
# Change the features
# ============================================================

print()
print()
print("=" * 70)
print("EXPERIMENT B: CHANGE THE FEATURES")
print("=" * 70)


# ------------------------------------------------------------
# Original feature set
# ------------------------------------------------------------
# This is the original set from the notebook.

features_original = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]


# ------------------------------------------------------------
# Alternative feature set
# ------------------------------------------------------------
# We DROP impressions_90d
# We ADD engagement_rate
#
# This lets us see whether the tree learns a different rule
# when we change the information available to it.

features_alternative = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]


# ------------------------------------------------------------
# Function to prepare feature data
# ------------------------------------------------------------
# This prevents us from repeating the same cleaning code.

def prepare_features(feature_list):

    return (
        df[feature_list]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )


# ------------------------------------------------------------
# Train a depth-2 tree using the ALTERNATIVE features
# ------------------------------------------------------------

X_alternative = prepare_features(features_alternative)

alternative_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

alternative_tree.fit(
    X_alternative,
    y
)


# ------------------------------------------------------------
# Calculate Precision@20 and Precision@50
# ------------------------------------------------------------

alternative_scores = alternative_tree.predict_proba(
    X_alternative
)[:, 1]

alternative_precision20 = precision_at_k(
    alternative_scores,
    y,
    20
)

alternative_precision50 = precision_at_k(
    alternative_scores,
    y,
    50
)


print()
print("Alternative feature set:")
print(features_alternative)

print()
print(f"Precision@20: {alternative_precision20:.3f}")
print(f"Precision@50: {alternative_precision50:.3f}")


# ------------------------------------------------------------
# Print the alternative tree
# ------------------------------------------------------------
# The first split printed here is the feature the model
# chose to use first.

print()
print("Readable tree using alternative features:")
print(
    export_text(
        alternative_tree,
        feature_names=features_alternative
    )
)


# ============================================================
# EXPERIMENT C
# Train/Test Split
# ============================================================
#
# The earlier experiments evaluated on the same data used
# to train the model.
#
# That is called IN-SAMPLE evaluation.
#
# Here we separate the data into:
#
# 80% -> training
# 20% -> testing
#
# The model never sees the test labels during training.
# ============================================================

print()
print()
print("=" * 70)
print("EXPERIMENT C: TRAIN/TEST SPLIT")
print("=" * 70)


# ------------------------------------------------------------
# Split the original dataset
# ------------------------------------------------------------
# test_size=0.20 means:
#
# 80% of pages -> training
# 20% of pages -> testing
#
# stratify=y keeps the proportion of declining/non-declining
# pages approximately similar in both groups.

X_train, X_test, y_train, y_test = train_test_split(
    X_original,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print()
print("Training pages:", len(X_train))
print("Testing pages:", len(X_test))


# ------------------------------------------------------------
# Train a depth-2 tree using ONLY the training data
# ------------------------------------------------------------

test_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

test_tree.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# Evaluate on the TRAINING data
# ------------------------------------------------------------
# This tells us how well the model performs on data
# it has already seen.

train_scores = test_tree.predict_proba(
    X_train
)[:, 1]

train_precision20 = precision_at_k(
    train_scores,
    y_train,
    20
)

train_precision50 = precision_at_k(
    train_scores,
    y_train,
    50
)


# ------------------------------------------------------------
# Evaluate on the TESTING data
# ------------------------------------------------------------
# This is more useful for checking generalization because
# these pages were NOT used to train the model.

test_scores = test_tree.predict_proba(
    X_test
)[:, 1]

test_precision20 = precision_at_k(
    test_scores,
    y_test,
    20
)

test_precision50 = precision_at_k(
    test_scores,
    y_test,
    50
)


# ------------------------------------------------------------
# Print train vs test results
# ------------------------------------------------------------

print()
print("Depth-2 Decision Tree")
print("-" * 40)

print(
    f"TRAIN Precision@20: {train_precision20:.3f}"
)

print(
    f"TEST  Precision@20: {test_precision20:.3f}"
)

print()

print(
    f"TRAIN Precision@50: {train_precision50:.3f}"
)

print(
    f"TEST  Precision@50: {test_precision50:.3f}"
)


# ------------------------------------------------------------
# Print the tree learned from the training data
# ------------------------------------------------------------

print()
print("Tree learned from training data:")
print(
    export_text(
        test_tree,
        feature_names=original_features
    )
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print()
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print()

print("1. TREE DEPTH EXPERIMENT")
print(depth_results_df.to_string(index=False))

print()

print("2. ALTERNATIVE FEATURES")
print(
    f"Precision@20: {alternative_precision20:.3f}"
)

print(
    f"Precision@50: {alternative_precision50:.3f}"
)

print()

print("3. TRAIN vs TEST")
print(
    f"Train Precision@50: {train_precision50:.3f}"
)

print(
    f"Test  Precision@50: {test_precision50:.3f}"
)

print()

print("Done!")

Dataset shape: (30000, 6)
Number of pages: 30000

EXPERIMENT A: TEST DIFFERENT TREE DEPTHS

--- Decision Tree: max_depth=2 ---
Precision@20: 0.550
Precision@50: 0.600

Readable tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0


--- Decision Tree: max_depth=3 ---
Precision@20: 0.700
Precision@50: 0.720

Readable tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |

### Your Experiment

I tested decision trees with `max_depth` values of 2, 3, and 4. The Precision@50 results were **0.600**, **0.720**, and **0.680**, respectively. The depth-3 tree achieved the highest Precision@50 among the three settings I tested, improving from 0.600 at depth 2 to 0.720 at depth 3. Increasing the depth to 4 reduced Precision@50 to 0.680. The depth-3 tree was still reasonably readable, while the depth-4 tree produced more conditions and became more complex to interpret.

I also changed the feature set by dropping `impressions_90d` and adding `engagement_rate`. With the alternative feature set, the model achieved **0.850 Precision@20** and **0.700 Precision@50**. The first split changed from `impressions_90d` in the original model to **`avg_position`**, showing that changing the available features can change the rule learned by the tree.

Finally, I performed a train/test split with 24,000 pages for training and 6,000 pages for testing. The depth-2 tree achieved a training Precision@50 of **0.520** and a test Precision@50 of **0.620**. In this particular split, the test score was higher than the training score, showing that evaluation results can vary depending on the data used. This simple random split is only a teaching exercise; the actual pipeline uses client-holdout validation so that pages from the same client do not appear in both training and testing.

Overall, the experiment showed that tree depth, feature selection, and evaluation methodology can substantially affect the results. A more complex model is not automatically better, and changing the available features can cause the model to learn a different decision rule. The experiment also reinforced the importance of evaluating models on data that was not used for training and avoiding leakage.


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.